# Google Colab compatibility check

**Phase 1 · TP-01**

In Colab, upload this notebook with **File → Upload notebook** and select your existing Python 3.13 hosted runtime with GPU access. The smoke model runs on CPU so the exported predictions can be compared with Windows CPU.

Run the cells in order. They check the interpreter and installed package versions. If a version differs, record it and pause for review before changing the runtime. The notebook does not install packages.

In [ ]:
import platform
import sys

print("Python:", platform.python_version())
print("Kernel interpreter:", sys.executable)
print("Platform:", platform.platform())
if sys.version_info[:2] != (3, 13):
    raise RuntimeError("TP-01 requires a Python 3.13 runtime.")

In [ ]:
import hashlib
import importlib.metadata
import json
import subprocess
from pathlib import Path

import numpy as np
import tensorflow as tf

expected_versions = {"tensorflow": "2.21.0", "keras": "3.13.2", "numpy": "2.4.6"}
versions = {name: importlib.metadata.version(name) for name in expected_versions}
print("Installed versions:", versions)
if versions != expected_versions:
    raise RuntimeError(f"TP-01 version mismatch. Expected {expected_versions}.")
print("TensorFlow Keras version:", tf.keras.version())
print("CPU:", tf.config.list_physical_devices("CPU"))
print("GPU:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("CPU")
assert tf.keras.version() == importlib.metadata.version("keras")

dependency_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True, text=True, check=False,
)
print(f"pip check exit code: {dependency_check.returncode}")
print(dependency_check.stdout.strip() or dependency_check.stderr.strip())

In [ ]:
tolerance = 1e-6
artifact_dir = Path("artifacts/phase1/colab")
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / "phase1_smoke.keras"
reference_path = artifact_dir / "phase1_reference.json"
with tf.device("/CPU:0"):
    inputs = tf.constant([[1.0, 2.0]], dtype=tf.float32)
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(2,)),
        tf.keras.layers.Dense(2, activation="softmax"),
    ])
    model.layers[0].set_weights([
        np.array([[0.25, 0.5], [-0.5, 0.25]], dtype=np.float32),
        np.array([0.25, -0.25], dtype=np.float32),
    ])

    scores_before = model(inputs, training=False).numpy()
    model.save(model_path)
    reloaded = tf.keras.models.load_model(model_path, compile=False)
    scores_after = reloaded(inputs, training=False).numpy()

assert scores_before.shape == scores_after.shape == (1, 2)
assert np.isfinite(scores_before).all() and np.isfinite(scores_after).all()
np.testing.assert_allclose(scores_after, scores_before, rtol=0, atol=tolerance)
reference = {
    "python": platform.python_version(),
    **versions,
    "model_sha256": hashlib.sha256(model_path.read_bytes()).hexdigest(),
    "scores": scores_after.tolist(),
}
reference_path.write_text(json.dumps(reference, indent=2) + "\n", encoding="utf-8")

print("Scores before:", scores_before.tolist())
print("Scores after:", scores_after.tolist())
print("Maximum absolute difference:", float(np.max(np.abs(scores_after - scores_before))))
print("Same-runtime .keras agreement: PASS")
print("Download both files from:", artifact_dir.resolve())

## Evidence to record

Record Python and dependency versions, runtime selection, CPU and GPU visibility, the `pip check` output, and the maximum absolute reload difference. Report dependency conflicts separately from serialization agreement. Download `phase1_smoke.keras` and `phase1_reference.json` from the printed folder to run the Windows portability test described in the project README. Use only the pair generated by this run. The test does not train or use project data.